# IRCTC Business Insights
## An end-to-end data analyst portfolio

**Question:** How do bookings and recorded booking value vary by month, channel, type and route, and which conclusions does the source actually support?

This synthetic educational sample was provided with DataCliq tutorial and design references. The analysis does not represent official IRCTC performance. The project demonstrates source profiling, an explicit exception policy, relational modeling, SQL, visualization and reproducible validation.

This notebook uses Python's standard library. Its saved output was executed against the included source and SQLite database. Open the interactive dashboard separately for filters and charts.


In [1]:
from pathlib import Path
import csv, json, sqlite3, statistics
from collections import Counter, defaultdict
from datetime import date

ROOT = Path.cwd()
if not (ROOT / "data/raw/bookings.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "data/raw/bookings.csv").exists(), "Run from the project or notebooks folder"
raw = list(csv.DictReader((ROOT / "data/raw/bookings.csv").open()))
clean = list(csv.DictReader((ROOT / "data/processed/bookings_clean.csv").open()))
print(f"Raw rows: {len(raw):,}; accepted rows: {len(clean):,}")
print(f"Raw fields: {len(raw[0])}; accepted fields: {len(clean[0])}")


Raw rows: 20,000; accepted rows: 19,301
Raw fields: 28; accepted fields: 42


## 1. Profile before choosing the grain

A row is a booking, not a passenger or a train service. `Passenger_Count` can exceed one. A repeated booking ID cannot be resolved safely using a random date or the first row encountered.


In [2]:
original_columns = list(raw[0])
exact_counts = Counter(tuple(row[c] for c in original_columns) for row in raw)
exact_excess = sum(n - 1 for n in exact_counts.values())
deduplicated = [dict(zip(original_columns, values)) for values in exact_counts]
id_counts = Counter(row["Booking_ID"] for row in deduplicated)
conflict_ids = {key for key, n in id_counts.items() if n > 1}
conflict_rows = [r for r in deduplicated if r["Booking_ID"] in conflict_ids]
print(f"Exact duplicate excess: {exact_excess}")
print(f"Conflicting IDs: {len(conflict_ids):,}; quarantined rows: {len(conflict_rows):,}")
print(f"Accepted calculation: {len(raw):,} - {exact_excess} - {len(conflict_rows):,} = {len(clean):,}")
assert len(raw) - exact_excess - len(conflict_rows) == len(clean)
assert len({r["booking_id"] for r in clean}) == len(clean)


Exact duplicate excess: 1
Conflicting IDs: 349; quarantined rows: 698
Accepted calculation: 20,000 - 1 - 698 = 19,301


## 2. Reconcile amounts, not just counts

The source `Total_Revenue_INR` becomes `booking_value_inr`. Refunds and recognized-revenue rules are absent, so the sum is described as recorded booking value across all statuses.


In [3]:
raw_value = sum(int(r["Total_Revenue_INR"]) for r in raw)
unique_value = sum(int(r["Total_Revenue_INR"]) for r in deduplicated)
removed_value = raw_value - unique_value
conflict_value = sum(int(r["Total_Revenue_INR"]) for r in conflict_rows)
accepted_value = sum(int(r["booking_value_inr"]) for r in clean)
print(f"INR {raw_value:,} - INR {removed_value:,} - INR {conflict_value:,} = INR {accepted_value:,}")
assert raw_value - removed_value - conflict_value == accepted_value
components = ["Base_Fare_INR", "Surge_Tatkal_Charges_INR", "Catering_Charges_INR", "Taxes_And_Convenience_INR"]
fare_errors = sum(sum(int(r[c]) for c in components) != int(r["Total_Revenue_INR"]) for r in raw)
print("Raw fare component mismatches:", fare_errors)
assert fare_errors == 0


INR 53,055,565 - INR 660 - INR 1,775,602 = INR 51,279,303
Raw fare component mismatches: 0


## 3. Define the date limitation

Invalid chronology stays in descriptive booking-date summaries as a visible flag. It is excluded from lead-time measures only. We do not overwrite dates or claim demand seasonality is trustworthy.


In [4]:
lead_times = [(date.fromisoformat(r["travel_date"]) - date.fromisoformat(r["booking_date"])).days for r in clean]
valid_leads = [v for v in lead_times if v >= 0]
invalid = len(lead_times) - len(valid_leads)
print(f"Booked after travel: {invalid:,} / {len(clean):,} ({invalid/len(clean):.2%})")
print(f"Valid lead-time denominator: {len(valid_leads):,}")
print(f"Mean valid lead time: {statistics.mean(valid_leads):.2f} days")
print(f"Median valid lead time: {statistics.median(valid_leads):.1f} days")


Booked after travel: 11,408 / 19,301 (59.11%)
Valid lead-time denominator: 7,893
Mean valid lead time: 77.74 days
Median valid lead time: 77.0 days


## 4. Query the relational model

The normalized dimensions hold attributes only when the source supports their grain. Train number does not uniquely identify a train name or route in this synthetic file. Those fields remain on the booking record.


In [5]:
db = sqlite3.connect(ROOT / "data/irctc.sqlite")
db.row_factory = sqlite3.Row
print("Tables and views:")
for r in db.execute("SELECT type, name FROM sqlite_master WHERE type IN ('table','view') ORDER BY type,name"):
    print(f"  {r['type']:5s} {r['name']}")

def query(sql):
    cursor = db.execute(sql)
    data = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    widths = [max(len(str(c)), *(len(str(r[i])) for r in data)) for i,c in enumerate(columns)]
    print(" | ".join(c.ljust(widths[i]) for i,c in enumerate(columns)))
    print("-+-".join("-"*w for w in widths))
    for row in data:
        print(" | ".join(str(value).ljust(widths[i]) for i,value in enumerate(row)))


Tables and views:
  table dim_date
  table dim_route
  table dim_train
  table fact_booking
  view  v_channels
  view  v_monthly
  view  v_routes
  view  v_status


In [6]:
query("""
SELECT route,
       COUNT(*) AS bookings,
       SUM(booking_value_inr) AS recorded_value_inr,
       ROUND(AVG(booking_value_inr), 2) AS avg_value_inr
FROM fact_booking
GROUP BY route
ORDER BY recorded_value_inr DESC
""")


route                 | bookings | recorded_value_inr | avg_value_inr
----------------------+----------+--------------------+--------------
Delhi - Mumbai        | 4245     | 18405394           | 4335.78      
Delhi - Howrah        | 3442     | 15444108           | 4486.96      
Lucknow - Delhi       | 2315     | 4177128            | 1804.37      
Bengaluru - Chennai   | 2681     | 3665106            | 1367.07      
Mumbai - Pune         | 2890     | 2574272            | 890.75       
Howrah - Puri         | 1386     | 2529705            | 1825.18      
Ahmedabad - Mumbai    | 1390     | 2460329            | 1770.02      
Hyderabad - Bengaluru | 952      | 2023261            | 2125.27      


In [7]:
query("""
SELECT booking_channel, booking_type,
       COUNT(*) AS bookings,
       SUM(booking_value_inr) AS recorded_value_inr
FROM fact_booking
GROUP BY booking_channel, booking_type
ORDER BY booking_channel, recorded_value_inr DESC
""")


booking_channel     | booking_type   | bookings | recorded_value_inr
--------------------+----------------+----------+-------------------
Authorised Partners | Reserved       | 1703     | 4087656           
Authorised Partners | Tatkal         | 381      | 1060770           
Authorised Partners | Premium Tatkal | 255      | 887793            
IRCTC App           | Reserved       | 4847     | 11833915          
IRCTC App           | Tatkal         | 1069     | 3100321           
IRCTC App           | Premium Tatkal | 726      | 2605121           
IRCTC Website       | Reserved       | 7484     | 18577774          
IRCTC Website       | Tatkal         | 1712     | 5120288           
IRCTC Website       | Premium Tatkal | 1124     | 4005665           


In [8]:
query("""
WITH monthly AS (
  SELECT booking_month, COUNT(*) AS bookings,
         SUM(booking_value_inr) AS recorded_value_inr
  FROM fact_booking GROUP BY booking_month
), comparison AS (
  SELECT *, LAG(bookings) OVER (ORDER BY booking_month) AS previous_bookings
  FROM monthly
)
SELECT booking_month, bookings, recorded_value_inr,
       ROUND(100.0 * (bookings - previous_bookings) / NULLIF(previous_bookings, 0), 2) AS mom_pct
FROM comparison ORDER BY booking_month
""")


booking_month | bookings | recorded_value_inr | mom_pct
--------------+----------+--------------------+--------
2025-01       | 1428     | 3774093            | None   
2025-02       | 1390     | 3623607            | -2.66  
2025-03       | 1518     | 3881082            | 9.21   
2025-04       | 1461     | 3920984            | -3.75  
2025-05       | 1627     | 4467616            | 11.36  
2025-06       | 1657     | 4464593            | 1.84   
2025-07       | 1418     | 3899307            | -14.42 
2025-08       | 1539     | 3974512            | 8.53   
2025-09       | 1655     | 4356748            | 7.54   
2025-10       | 1783     | 4811471            | 7.73   
2025-11       | 2010     | 5103595            | 12.73  
2025-12       | 1815     | 5001695            | -9.7   


## 5. Independently verify the headline controls

Checks compare the CSV to SQL rather than merely comparing labels with hardcoded expected strings. The full test suite also covers duplicate policies and edge cases.


In [9]:
sql_control = db.execute("SELECT COUNT(*) n, SUM(booking_value_inr) v, SUM(passenger_count) p, COUNT(DISTINCT user_id) u, COUNT(DISTINCT train_number) t FROM fact_booking").fetchone()
csv_control = (len(clean), accepted_value, sum(int(r["passenger_count"]) for r in clean), len({r["user_id"] for r in clean}), len({r["train_number"] for r in clean}))
assert tuple(sql_control) == csv_control
print(dict(zip(["bookings", "recorded_value_inr", "passengers", "distinct_users", "distinct_train_numbers"], csv_control)))
print("CSV-to-SQL control reconciliation: PASS")
db.close()


{'bookings': 19301, 'recorded_value_inr': 51279303, 'passengers': 38528, 'distinct_users': 19101, 'distinct_train_numbers': 540}
CSV-to-SQL control reconciliation: PASS


## 6. Explain findings responsibly

Use `docs/CASE_STUDY.md` and the executive report for the completed observations and recommended next investigations. Route concentration, channel differences and status mix are descriptive patterns in this sample. They do not establish campaign impact, profitability, customer preference or real operational reliability.

For a production extension, obtain authoritative booking events, refund and settlement ledgers, a reliable train-service key, seat inventory and customer history. Define the decision and validation dataset before forecasting or recommending resource allocation.

## Reproduce and present

Run `python3 src/pipeline.py`, the tests documented in the project README, and `python3 src/build_dashboard.py`. Open the offline HTML dashboard. Present the business question, the duplicate decision, two quantified observations and the boundaries of the evidence. The Power BI kit is supplied separately and still requires native-engine validation.
